# Machine learning in production: the imbalance reveal

Exploration only. Everything the service depends on lives in `nw/triage`; this notebook imports it. Open it from `notebooks/` with the project environment; the paths below are relative to that directory.

In [1]:
import json, collections
from pathlib import Path
import numpy as np

from nw.triage.train import (
    load_tickets,
    by_split,
    labels,
    build_pipeline,
    evaluate,
    choose_p0_threshold,
)
from nw.triage.model import TriageModel
from nw.triage.features import PRIORITIES

rows = load_tickets(Path("../data/tickets.jsonl"))
splits = by_split(rows)
len(rows), {k: len(v) for k, v in splits.items()}

(7950, {'train': 6361, 'val': 795, 'test': 794})

## What the label looks like

In [2]:
counts = collections.Counter(r["priority"] for r in rows)
for p in PRIORITIES:
    print(f"{p}: {counts[p]:5d}  {100 * counts[p] / len(rows):5.1f}%")

P0:   314    3.9%
P1:  2571   32.3%
P2:  3295   41.4%
P3:  1770   22.3%


## The naive model

No class weights, plain argmax. Watch the P0 row of the confusion matrix.

In [3]:
naive = build_pipeline(class_weight=None).fit(splits["train"], labels(splits["train"]))
m = TriageModel(
    pipeline=naive,
    classes=list(naive.named_steps["clf"].classes_),
    p0_threshold=1.01,
    metadata={"version": "naive"},
)
rep = evaluate(m, splits["test"])
print(
    f"accuracy {rep['accuracy']:.3f}  macro-F1 {rep['macro_f1']:.3f}  P0 recall {rep['p0_recall']:.3f}"
)
for label, row in zip(PRIORITIES, rep["confusion"]):
    print(label, row)

accuracy 0.694  macro-F1 0.688  P0 recall 0.581
P0 [18, 13, 0, 0]
P1 [1, 192, 59, 5]
P2 [1, 40, 245, 43]
P3 [0, 3, 78, 96]


## Class weights

In [4]:
weighted = build_pipeline(class_weight="balanced").fit(splits["train"], labels(splits["train"]))
m = TriageModel(
    pipeline=weighted,
    classes=list(weighted.named_steps["clf"].classes_),
    p0_threshold=1.01,
    metadata={"version": "weighted"},
)
rep = evaluate(m, splits["test"])
print(
    f"accuracy {rep['accuracy']:.3f}  macro-F1 {rep['macro_f1']:.3f}  P0 recall {rep['p0_recall']:.3f}"
)
for label, row in zip(PRIORITIES, rep["confusion"]):
    print(label, row)

accuracy 0.696  macro-F1 0.717  P0 recall 0.774
P0 [24, 7, 0, 0]
P1 [3, 196, 52, 6]
P2 [2, 45, 220, 62]
P3 [0, 2, 62, 113]


## The threshold trade-off

Recall and precision for P0 as the threshold moves, on the validation split. Pick the point Northwind can live with.

In [5]:
val = splits["val"]
proba = m.predict_proba(val)[:, 0]
y = labels(val) == "P0"
print(" t    recall  precision  flagged")
for t in np.linspace(0.05, 0.95, 19):
    pred = proba >= t
    rec = (pred & y).sum() / max(1, y.sum())
    prec = (pred & y).sum() / max(1, pred.sum())
    print(f"{t:.2f}   {rec:.2f}     {prec:.2f}     {pred.sum():4d}")
print("chosen:", choose_p0_threshold(proba, labels(val)))

 t    recall  precision  flagged
0.05   0.94     0.29      100
0.10   0.90     0.39       71
0.15   0.87     0.53       51
0.20   0.84     0.54       48
0.25   0.84     0.59       44
0.30   0.81     0.61       41
0.35   0.81     0.62       40
0.40   0.81     0.71       35
0.45   0.81     0.74       34
0.50   0.81     0.78       32
0.55   0.77     0.83       29
0.60   0.74     0.88       26
0.65   0.74     0.88       26
0.70   0.74     0.92       25
0.75   0.71     0.92       24
0.80   0.65     0.91       22
0.85   0.61     0.90       21
0.90   0.58     0.95       19
0.95   0.39     1.00       12
chosen: 0.13


## Calibration

Are the probabilities honest? This cell loads the artifact `make train-triage` last wrote, so rerun it after each retrain. Bucket the top probability and compare with how often the top guess was right; the gap, weighted by bucket size, is the expected calibration error (ECE) in the report.

In [6]:
m = TriageModel.load(Path("../artifacts/triage/latest"))
proba_all = m.predict_proba(splits["test"])
top = proba_all.max(axis=1)
correct = np.asarray([PRIORITIES[int(np.argmax(p))] for p in proba_all]) == labels(splits["test"])
for lo in np.arange(0.2, 1.0, 0.1):
    mask = (top > lo) & (top <= lo + 0.1)
    if mask.any():
        print(
            f"confidence {lo:.1f} to {lo + 0.1:.1f}: predicted {top[mask].mean():.2f}, actual {correct[mask].mean():.2f}, n={mask.sum()}"
        )

confidence 0.3 to 0.4: predicted 0.38, actual 0.58, n=12
confidence 0.4 to 0.5: predicted 0.47, actual 0.56, n=121
confidence 0.5 to 0.6: predicted 0.55, actual 0.61, n=255
confidence 0.6 to 0.7: predicted 0.65, actual 0.75, n=216
confidence 0.7 to 0.8: predicted 0.75, actual 0.84, n=126
confidence 0.8 to 0.9: predicted 0.84, actual 0.98, n=59
confidence 0.9 to 1.0: predicted 0.92, actual 1.00, n=5


## Questions to bring back to the room

- Which threshold would you defend to the on-call lead, and with what number?
- Where do the P0 misses go in the confusion matrix, and what do those tickets have in common?